# Macro Quant — bac à sable

Env : `conda activate macro-quant` (kernel **Python (macro-quant)**).

**Attention** : `macro_quant_engine.py` n'a pas de garde `__main__` — l'importer **exécute tout le pipeline**
(fetch FRED/Yahoo, features, PCA, analogues, base rates). Premier run ~10–20 s, ensuite cache `/tmp/fredcache`.
L'avantage : après l'import, tous les objets internes sont vivants dans `mq`.

In [ ]:
import sys, os, json, pathlib

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "engine"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

print(ROOT)

In [ ]:
# Lance le pipeline complet (verbeux). Re-run = re-exécution : préfère recharger le JSON plus bas.
import macro_quant_engine as mq

## Ce que `mq` expose

| objet | contenu |
|---|---|
| `mq.SERIES`, `mq.YAHOO` | univers de séries (id → nom/type) |
| `mq.cal` | calendrier commun (dates `YYYY-MM-DD`) |
| `mq.LV` | niveaux alignés `{sid: array}` |
| `mq.RET` | rendements quotidiens natifs (log pour prix, bps pour taux) |
| `mq.Z`, `mq.FEATNAMES` | matrice de features z-scorées (causales, expanding) |
| `mq.vidx`, `mq.today_v` | index des jours valides / ligne du jour |
| `mq.D`, `mq.K_ANALOG` | distances Mahalanobis, nb d'analogues |
| `mq.fwd`, `mq.stationary_bootstrap_ci` | forward returns, IC block bootstrap |

In [ ]:
# Features du jour
z_today = mq.Z[mq.today_v]
pd.Series(z_today, index=mq.FEATNAMES, name=f"z @ {mq.cal[mq.vidx[mq.today_v]]}").round(2).to_frame()

In [ ]:
# Les k analogues historiques les plus proches du régime du jour
order = np.argsort(mq.D)
ana = [(mq.cal[mq.vidx[i]], float(mq.D[i])) for i in order[: mq.K_ANALOG]]
df_ana = pd.DataFrame(ana, columns=["date", "maha"])
print(f"{len(df_ana)} analogues, rayon max = {df_ana.maha.max():.2f}")
df_ana.head(15)

In [ ]:
# Où tombent les analogues dans le temps ? (concentration = régime daté, pas structurel)
yrs = pd.to_datetime(df_ana.date).dt.year.value_counts().sort_index()
ax = yrs.plot.bar(figsize=(11, 3), color="#4C78A8")
ax.set_title("Analogues par année"); ax.set_ylabel("n jours")
plt.tight_layout(); plt.show()

In [ ]:
# Rapport du dernier run (écrit par l'engine dans /tmp)
rep = json.load(open("/tmp/macro_quant_report.json"))
print(rep["asof"], "| n_analog", rep["n_analog"], "| radius", round(rep["maha_radius"], 2))

# assets = {sid: {name, kind, h: {"5"|"10"|"20": stats}}}
df_rep = pd.DataFrame([
    {"sid": sid, "asset": a["name"], "kind": a["kind"], "h": int(h), **st}
    for sid, a in rep["assets"].items()
    for h, st in a["h"].items()
])
df_rep[df_rep.h == 10].sort_values("lift_pneg", key=abs, ascending=False).head(10)

## Base rates historisés (`db/`)

Chaque run de `engine/macro_quant_daily.py` append une ligne. Schéma : `db/SCHEMA.md`.

In [ ]:
br = pd.read_csv("db/base_rates.csv")
rf = pd.read_csv("db/regime_features.csv")
print(br.shape, rf.shape, "| runs:", sorted(br.run_date.unique()))

last = br[(br.run_date == br.run_date.max()) & (br.h == 10)]
last.sort_values("lift_pneg", key=abs, ascending=False)[
    ["name", "mean_cond", "mean_uncond", "lift_mean", "pneg_cond", "lift_pneg", "n_eff", "tag", "signal"]
].head(15)

In [ ]:
# Garde-fou n_eff : sous ~10, l'IC bootstrap est du bruit (fenêtres forward chevauchantes)
ax = last.set_index("name").n_eff.sort_values().plot.barh(figsize=(7, 7), color="#72B7B2")
ax.axvline(10, color="crimson", ls="--", lw=1, label="n_eff = 10")
ax.set_title("n_eff par asset (h=10j)"); ax.legend()
plt.tight_layout(); plt.show()

## Autres entrées

```python
import qstat            # adf, kpss, half_life, engle_granger, kalman_hedge, benjamini_hochberg
import yfetch           # yfetch.fetch("CL=F") -> {date: close}, caché /tmp/yfcache
```

Scripts (en terminal, env activé) :
```bash
python engine/macro_quant_backtest.py   # backtest + DSR / PBO / hold-out
python engine/macro_quant_daily.py      # persiste le run du jour dans db/
python engine/make_daily_dashboard.py   # figures -> results/figures/
```